# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **three trigger variants**:
- `score_threshold`  — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`    — every open-buy event (fixed stake baseline)
- `copy_triggers`    — every trade (open/add/close/reduce), tight slippage


In [593]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [594]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [595]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-04-15  TRAIN_A_END_DATE=2025-12-24


## Compute / load wallet skill metrics

In [596]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [597]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [598]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

## Build wallet profiles and signal events

In [599]:
# from polymarket_analysis.signal.builder import (
#     build_wallet_profiles,
#     build_signal_events,
# )

# selected_wallet_profiles = build_wallet_profiles(
#     dataset, selected_wallets, period="full_train",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
# )
# selected_wallet_profiles.to_parquet(
#     WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
# )

# # Force-regenerate signal caches: the schema now includes all event types
# # (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# # copy_token_winner columns. Delete stale parquets so they are always rebuilt.
# CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
# TEST_SIGNALS_PATH.unlink(missing_ok=True)

# _, train_b_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="train_b",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

# _, test_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="test",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

# print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
# print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

## Calibrate signal scoring on train-B

In [600]:
# from polymarket_analysis.signal.scorer import (
#     build_calibration_tables,
#     apply_signal_score,
#     select_signal_threshold,
# )

# # Calibration tables are built from open_buy events only — the scoring model
# # is designed for open-buy conviction features. Non-open-buy rows get neutral
# # defaults and will not be used to calibrate the scorer.
# train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

# price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
# train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
# SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
# print(f"Global edge: {global_edge:.4f}")
# print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# # score distribution
# score_deciles = (
#     train_b_scored.assign(
#         score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
#     )
#     .groupby("score_decile")
#     .agg(
#         count=("signal_score", "size"),
#         avg_signal_score=("signal_score", "mean"),
#         avg_copy_roi_capped=("copy_roi_capped", "mean"),
#     )
#     .reset_index()
# )
# score_deciles

## Build wallet cohorts (skill path)

In [601]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

# wallet_cohorts = build_wallet_cohorts(
#     full_train_metrics, train_b_open_buys, selected_wallets,
#     top_n=BEST_TOP_N,
# )
# {name: len(df) for name, df in wallet_cohorts.items()}


## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [602]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"]      = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"]         = df_full["quantity"].astype(float)

#filter only buy
df_full = df_full[df_full["side"] == "BUY"].copy()

df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)
# ignore buys with large price, so we don't skew roi
# df_full = df_full[df_full['price'] <= 0.95]
df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))
# del df_full, df_slice

18421


In [603]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'], 0, 1.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
# wallet_vol_train = wallet_vol_train[wallet_vol_train['copyable_roi'] > 0.05]
# wallet_vol_train = wallet_vol_train[wallet_vol_train['top_market_pnl_pct'] < 0.3]

In [604]:
wallet_cohorts = {}

In [605]:
print(len(wallet_vol_train))
wallet_vol_train.head()


18421


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1.52,150.48,150.48,1.00,1.00,1.00,99.00,2026-02-27 06:10:00+00:00,99.00,0.00,0.00,0.00,0.00,99.00,1.00,99.00
1,0x8da0d29ec66508950ad72846ac3164db7cc41c53,NaN,1,1,2.50,205.83,205.83,1.00,1.00,1.00,82.33,2026-04-03 19:40:00+00:00,82.33,0.00,0.00,0.00,0.00,82.33,1.00,82.33
2,0xd7ca8219c8afa07b455ab7e004fc5381b3727b1e,1.83,3,1,5.60,374.40,374.40,1.00,1.00,1.00,73.26,2026-04-04 16:40:00+00:00,78.25,0.00,0.00,0.00,0.00,66.83,1.00,78.25
3,0x9dd6c4bc2c94a2eea4ecf6148bf3cf63a725ad7b,NaN,1,1,3.15,206.85,206.85,1.00,1.00,1.00,65.67,2026-02-26 16:10:00+00:00,65.67,0.00,0.00,0.00,0.00,65.67,1.00,65.67
4,0xcbf45e4a1de6f54869fedf2ab7c064f248948723,1.72,10,4,643.81,1958.70,1762.99,1.00,1.00,0.99,82.33,2026-04-04 13:12:30+00:00,59.22,0.00,0.00,0.00,0.00,3.04,0.90,53.30


In [606]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

# filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['0x0a'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

# print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# # Build wallet_quality based on total_pnl rank (higher = better)
# filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
# filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
#     method="first", pct=True
# )

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
# vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
#     set(filtered_wallets_vol["wallet"])
# )
# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

In [607]:
# wallet_cohorts['multi_filter'] = wallet_vol_train[
#     (wallet_vol_train['copyable_roi'] >= 0.04)
#     & (wallet_vol_train['num_buckets'] >= 20)
#     &(wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
#     &(wallet_vol_train['copyable_pnl_factor'] >=0.4)
#     # & (wallet_vol_train['median_roi'] >= 0)
#     & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.3)
#     & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
#     # & (wallet_vol_train['num_markets'] >= 100)
#     & (wallet_vol_train['top_market_pnl_pct'] < 0.4)
#     & (wallet_vol_train['top5_pnl_pct'] < 0.5)
#     # & (wallet_vol_train['total_pnl'] < 10_000)
# ].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

wallet_cohorts['multi_filter'] = wallet_vol_train[
    #(wallet_vol_train['copyable_roi'] >= 0.02)
    (wallet_vol_train['average_roi'] >= 0.05)
    & (wallet_vol_train['median_roi'] >= 0)
    # & (wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
    & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.2)
    # & (wallet_vol_train['max_drawdown_to_pnl'] >= 0.1)
    # & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
    & (wallet_vol_train['num_buckets'] >= 20)
    & (wallet_vol_train['top_market_pnl_pct'] < 0.5)
    & (wallet_vol_train['top5_pnl_pct'] < 0.5)

#    TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and date
    & (wallet_vol_train['median_dt'].dt.date <= (END_DATE_TRAIN - pd.Timedelta(days=60)))
    # & (wallet_vol_train['copyable_pnl'] >= 100)
    # & (wallet_vol_train['copyable_pnl_factor'] >= 0.1)
    # & (wallet_vol_train['total_pnl'] > 10_000)
].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

len(wallet_cohorts['multi_filter'])

147

In [608]:
END_DATE_TRAIN - pd.Timedelta(days=60)

datetime.date(2026, 2, 14)

In [659]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['dt'] > pd.to_datetime("2026-04-16", utc=True))
    # & (df_full['wallet'] == '0xbacd00c9080a82ded56f504ee8810af732b0ab35')
    & (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(20)

-392339.24860010133
199696.04631756913


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
932,"Will Trump say ""Iran"" during events with Xi Jinping?",0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,Yes,-7946.50,18,11,93352.01,0.09
933,"Will Trump say ""Iran"" during events with Xi Jinping?",0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,No,85405.51,355,18,93352.01,0.09
2405,"US x Iran diplomatic meeting by April 22, 2026?",0xa573400b588079857899fce1e5a68da2d086f63fd35cbdd626df397e19e9995e,No,3742.42,62,9,52094.86,0.07
2404,"US x Iran diplomatic meeting by April 22, 2026?",0xa573400b588079857899fce1e5a68da2d086f63fd35cbdd626df397e19e9995e,Yes,-48352.44,199,12,52094.86,0.07
817,"Will Trump say ""Nuclear"" during events with Xi Jinping?",0x7f4378c2e8fb0a4673813e5a5d6f5a773765bab522186932a0465f326a05328d,Yes,-399.97,8,4,42180.09,0.01
816,"Will Trump say ""Nuclear"" during events with Xi Jinping?",0x7f4378c2e8fb0a4673813e5a5d6f5a773765bab522186932a0465f326a05328d,No,41780.12,111,12,42180.09,0.01
2460,Trump announces end of military operations against Iran by May 31st?,0x57c1e8de9d359a76055fe1be95e46a1e72d0537811dcc2ccf070cdfa73d8ba33,No,-92.32,7,4,31176.48,0.00
2461,Trump announces end of military operations against Iran by May 31st?,0x57c1e8de9d359a76055fe1be95e46a1e72d0537811dcc2ccf070cdfa73d8ba33,Yes,31084.17,58,11,31176.48,0.00
2624,"Iran agrees to surrender enriched uranium stockpile by April 30, 2026?",0x5d37825716832a4a54f89450932e89510f26cf4be59aeec3149d2c49e5fdf44d,No,24715.33,56,6,25782.20,0.04
2623,"Iran agrees to surrender enriched uranium stockpile by April 30, 2026?",0x5d37825716832a4a54f89450932e89510f26cf4be59aeec3149d2c49e5fdf44d,Yes,-1066.86,30,3,25782.20,0.04


In [610]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False).head(100)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
8,0x1521b47bf0c41f6b7fd3ad41cdec566812c8f23e,5.18,5511,169,25626952.46,2137669.21,512662.43,0.19,-0.02,0.20,0.08,2026-01-16 22:35:00+00:00,1.02,34908.62,0.02,11364.17,0.02,0.08,0.24,0.24
28,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,18.76,650,42,3593114.15,1263871.73,471312.30,0.45,-0.11,0.34,0.27,2025-10-27 08:50:00+00:00,0.36,62723.87,0.05,23756.96,0.05,0.35,0.37,0.13
102,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,9.98,1710,223,31473501.67,1182166.10,337889.12,0.28,-0.18,0.11,0.02,2025-03-20 18:12:30+00:00,0.11,106917.99,0.09,51088.45,0.15,0.04,0.29,0.03
60,0xe25b9180f5687aa85bd94ee309bb72a464320f1b,4.75,1464,245,10058765.12,665863.41,295399.04,0.24,-0.12,0.19,0.10,2025-09-12 19:20:00+00:00,0.15,56677.22,0.09,11527.62,0.04,0.07,0.44,0.07
56,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,8.21,2653,606,2555129.80,482435.39,211520.46,0.42,-0.15,0.36,0.05,2025-11-21 14:45:00+00:00,0.17,60441.60,0.13,22521.26,0.11,0.19,0.44,0.07
21,0x75049bd489194be19c45c31ed311e556411c9c69,11.29,650,65,4668426.15,632361.01,133510.95,0.47,-0.16,0.42,0.03,2025-06-22 13:22:30+00:00,0.73,92705.42,0.15,46004.81,0.34,0.14,0.21,0.15
114,0xc4d1a863e9cc45d02ba22d3a1ae9ba7822018ce8,4.86,905,74,17083434.78,395555.19,132903.87,0.32,-0.12,0.18,0.02,2025-11-05 05:45:00+00:00,0.06,46035.54,0.12,970.00,0.01,0.02,0.34,0.02
45,0xd748c701ad93cfec32a3420e10f3b08e68612125,4.48,1756,222,3824147.63,300795.39,119261.84,0.24,-0.21,0.23,0.11,2025-10-17 21:35:00+00:00,0.21,50499.01,0.17,27879.10,0.23,0.08,0.40,0.08
92,0x22e4248bdb066f65c9f11cd66cdd3719a28eef1c,4.54,2970,357,2970431.97,225924.49,99013.69,0.33,-0.14,0.12,0.08,2025-12-25 09:55:00+00:00,0.09,15954.92,0.07,6601.36,0.07,0.08,0.44,0.04
22,0x183a2a0b034877e761af0778da5134bebc37d514,13.80,487,69,426036.47,214117.85,94788.93,0.29,-0.21,0.40,0.25,2026-01-25 11:20:00+00:00,0.33,40810.17,0.19,9273.49,0.10,0.50,0.44,0.15


In [611]:
df_slice.head(1)

,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,...,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
0,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 15:56:11+00:00,BUY,1633.00,1633.00,0.87,1418.29,0.00,-1418.29,...,-0.00,0.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge, pardon]",Politics,110508738387624671992646538975675731011839126477460430600317798022691903999417,No,-1418.29,1418.29


In [612]:
wallets = set(wallet_cohorts['multi_filter']['wallet'])

df = df_full[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

620230
22203


In [613]:
MARKET = '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id                   0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20
end_date_iso                                                                 2026-01-31T00:00:00Z
question                                                    US strikes Iran by February 28, 2026?
tags                [Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]
primary_tag                                                                              Politics
winner_token_id    110790003121442365126855864076707686014650523258783405996925622264696084778807
outcome                                                                                       Yes
Name: 59763, dtype: object


In [614]:
dfc[(dfc['wallet'] == '0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8')
    & (dfc['quantity'] >= 1) ]['pnl']

Series([], Name: pnl, dtype: float64)

In [615]:
# ── positions per wallet + sum, dual-axis (position + price) ──────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(dfc) != 0:

    # ── 1. build per-bucket price series for YES (volume-weighted avg)
    _yes = dfc[dfc['outcome'] == 'Yes'].copy()
    _yes['_wv'] = _yes['avg_price'] * _yes['quantity']
    price_ts = (
        _yes.groupby('dt')[['_wv', 'quantity']].sum()
        .rename(columns={'quantity': '_qty'})
        .assign(vwap=lambda d: d['_wv'] / d['_qty'])[['vwap']]
        .reset_index()
        .sort_values('dt')
    )

    # ── 2. net YES position per wallet per timestamp (YES pos - NO pos)
    # position is cumulative after each trade; take last value per (dt, wallet, outcome)
    pos_per_wallet = (
        dfc
        .groupby(['dt', 'wallet', 'outcome'])['position']
        .last()
        .reset_index()
    )
    pos_yes = pos_per_wallet[pos_per_wallet['outcome'] == 'Yes'].rename(columns={'position': 'pos_yes'})[['dt', 'wallet', 'pos_yes']]
    pos_no = pos_per_wallet[pos_per_wallet['outcome'] == 'No'].rename(columns={'position': 'pos_no'})[['dt', 'wallet', 'pos_no']]
    net_pos = (
        pos_yes
        .merge(pos_no, on=['dt', 'wallet'], how='outer')
        .fillna(0)
    )
    net_pos['net'] = net_pos['pos_yes'] - net_pos['pos_no']
    net_pos = net_pos.sort_values(['wallet', 'dt']).reset_index(drop=True)

    # ── 3. weight wallets by proper probabilistic score (Brier skill) with shrinkage
    wallet_scores = (
        train_b_metrics[['wallet', 'brier_skill', 'open_buy_trades']]
        .drop_duplicates('wallet')
        .copy()
    )
    wallet_scores['score'] = wallet_scores['brier_skill'].clip(lower=0.0).fillna(0.0)
    wallet_scores['confidence'] = (
        wallet_scores['open_buy_trades'].fillna(0.0)
        / (wallet_scores['open_buy_trades'].fillna(0.0) + 25.0)
    )
    wallet_scores['wallet_weight'] = wallet_scores['score'] * wallet_scores['confidence']
    wallet_weights = wallet_scores.set_index('wallet')['wallet_weight']

    # ── 4. normalize by each wallet's typical move size in this market
    net_pos['net_step'] = net_pos.groupby('wallet')['net'].diff()
    typical_move = net_pos.groupby('wallet')['net_step'].apply(
        lambda s: s.abs().dropna().median() if s.notna().any() else np.nan
    ).rename('typical_move')
    positive_moves = typical_move[typical_move > 0]
    fallback_move = positive_moves.median() if not positive_moves.empty else 1.0
    typical_move = typical_move.fillna(fallback_move).clip(lower=fallback_move * 0.25)

    # ── 5. carry positions forward so the signal reflects current wallet stance
    timeline = price_ts['dt'].sort_values().drop_duplicates()
    net_panel = (
        net_pos.pivot(index='dt', columns='wallet', values='net')
        .sort_index()
        .reindex(timeline)
        .ffill()
        .fillna(0.0)
    )
    available_wallets = net_panel.columns.intersection(wallet_weights.index).intersection(typical_move.index)
    net_panel = net_panel[available_wallets]
    wallet_weights = wallet_weights.reindex(available_wallets).fillna(0.0)
    typical_move = typical_move.reindex(available_wallets)
    normalized_panel = net_panel.divide(typical_move, axis=1)
    weighted_panel = normalized_panel.multiply(wallet_weights, axis=1)

    signal_ts = pd.DataFrame({
        'dt': net_panel.index,
        'total_net': net_panel.sum(axis=1).to_numpy(),
        'weighted_position': weighted_panel.sum(axis=1).to_numpy(),
    })

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=['YES token (positive weighted signal => predicts YES)'],
        specs=[[{'secondary_y': True}]],
    )

    COLORS = [
        '#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66',
        '#77BEDB', '#F7A35C', '#90ED7D', '#8085E9', '#F15C80',
    ]

    top_wallets = net_panel.abs().max().sort_values(ascending=False).head(10).index

    # ── per-wallet net position (step: hold latest value, no interpolation)
    for c_idx, wallet in enumerate(top_wallets):
        wdf = (
            net_panel[[wallet]].reset_index()
            .rename(columns={wallet: 'net'})
            .sort_values('dt')
        )
        fig.add_trace(
            go.Scatter(
                x=wdf['dt'],
                y=wdf['net'],
                mode='lines',
                name=f'{wallet[:8]}...',
                line=dict(color=COLORS[c_idx % len(COLORS)], width=1, shape='hv'),
                legendgroup=wallet,
                opacity=0.6,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── total net position line (primary y, bold)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['total_net'],
            mode='lines',
            name='SUM (net YES)',
            line=dict(color='#222222', width=3, shape='hv'),
            legendgroup='sum_net',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── weighted normalized wallet position (primary y)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['weighted_position'],
            mode='lines',
            name='Weighted normalized position',
            line=dict(color='#C44E52', width=3, shape='hv'),
            legendgroup='weighted_signal',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── YES price line (secondary y)
    fig.add_trace(
        go.Scatter(
            x=price_ts['dt'],
            y=price_ts['vwap'],
            mode='lines+markers',
            name='Price (YES)',
            line=dict(color='#888888', width=2, dash='dot', shape='hv'),
            marker=dict(size=4),
            legendgroup='price_yes',
        ),
        row=1, col=1, secondary_y=True,
    )

    fig.add_hline(y=0, line=dict(color='#BBBBBB', width=1, dash='dash'))
    fig.update_yaxes(title_text='Net position / weighted normalized total', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Price (USDC)', row=1, col=1, secondary_y=True, range=[0, 1])

    fig.update_layout(
        title=f'Wallet net YES positions and weighted signal - {MARKET[:16]}...',
        height=500,
        template='plotly_white',
        legend=dict(orientation='v', x=1.05),
    )
    fig.show(renderer='browser')


In [616]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [617]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

147


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
150,0x1521b47bf0c41f6b7fd3ad41cdec566812c8f23e,Politics,26060642.76,0.77,1732782.94
159,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,Esports,5408881.64,1.00,1263871.73
282,0x1cc16713196d456f86fa9c7387dd326a7f73b8df,Politics,5073045.54,0.90,1049230.03
1864,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,Politics,28743443.68,0.80,909879.10
1293,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,Politics,3713458.60,0.76,552637.18
1216,0x8861f0bb5e0c19474ba73beeadc13ed8915beed6,Politics,2246730.06,1.00,518190.26
2097,0xe25b9180f5687aa85bd94ee309bb72a464320f1b,Politics,9056590.31,0.69,446026.22
1015,0x75049bd489194be19c45c31ed311e556411c9c69,Politics,2804038.03,0.43,395840.04
916,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,Politics,5629472.00,0.84,367586.14
1850,0xc4d1a863e9cc45d02ba22d3a1ae9ba7822018ce8,Politics,12374610.90,0.68,259901.90


In [618]:
# from polymarket_analysis.data.tags import load_tag_map
# from polymarket_analysis.data.tags import dominant_tag_stats

# split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# _result = dominant_tag_stats(
#     df_trades=df_full[df_full['dt'] >= split_dt],
#     wallets=wallet_cohorts["multi_filter"]["wallet"],
# )

# print(f"Wallets: {len(_result)}")
# _result[_result['primary_tag'] == 'Politics'].head(5)


In [619]:
# _result.groupby('primary_tag').agg(
#     tag_pnl=('tag_pnl', 'sum'),
#     wallets=('wallet', 'nunique')
#     )

In [620]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [621]:
print(len(wallet_cohorts['multi_filter']))
pd.set_option('display.max_rows', 100)
wallet_cohorts["multi_filter"].sort_values("total_pnl", ascending=False).head(30)

147


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
8,0x1521b47bf0c41f6b7fd3ad41cdec566812c8f23e,5.18,5511,169,25626952.46,2137669.21,512662.43,0.19,-0.02,0.20,0.08,2026-01-16 22:35:00+00:00,1.02,34908.62,0.02,11364.17,0.02,0.08,0.24,0.24
28,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,18.76,650,42,3593114.15,1263871.73,471312.30,0.45,-0.11,0.34,0.27,2025-10-27 08:50:00+00:00,0.36,62723.87,0.05,23756.96,0.05,0.35,0.37,0.13
102,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,9.98,1710,223,31473501.67,1182166.10,337889.12,0.28,-0.18,0.11,0.02,2025-03-20 18:12:30+00:00,0.11,106917.99,0.09,51088.45,0.15,0.04,0.29,0.03
23,0x1cc16713196d456f86fa9c7387dd326a7f73b8df,8.97,1589,174,1934051.78,1134810.93,71232.89,0.40,-0.09,0.32,0.06,2025-04-23 11:10:00+00:00,2.34,62530.54,0.06,25640.27,0.36,0.59,0.06,0.15
60,0xe25b9180f5687aa85bd94ee309bb72a464320f1b,4.75,1464,245,10058765.12,665863.41,295399.04,0.24,-0.12,0.19,0.10,2025-09-12 19:20:00+00:00,0.15,56677.22,0.09,11527.62,0.04,0.07,0.44,0.07
21,0x75049bd489194be19c45c31ed311e556411c9c69,11.29,650,65,4668426.15,632361.01,133510.95,0.47,-0.16,0.42,0.03,2025-06-22 13:22:30+00:00,0.73,92705.42,0.15,46004.81,0.34,0.14,0.21,0.15
16,0x8861f0bb5e0c19474ba73beeadc13ed8915beed6,16.67,294,17,1339542.15,518414.38,89665.49,0.45,-0.06,0.48,0.42,2025-04-04 01:12:30+00:00,1.00,25732.59,0.05,34.59,0.00,0.39,0.17,0.17
56,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,8.21,2653,606,2555129.80,482435.39,211520.46,0.42,-0.15,0.36,0.05,2025-11-21 14:45:00+00:00,0.17,60441.60,0.13,22521.26,0.11,0.19,0.44,0.07
114,0xc4d1a863e9cc45d02ba22d3a1ae9ba7822018ce8,4.86,905,74,17083434.78,395555.19,132903.87,0.32,-0.12,0.18,0.02,2025-11-05 05:45:00+00:00,0.06,46035.54,0.12,970.00,0.01,0.02,0.34,0.02
63,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,10.32,3712,1179,2619884.67,394721.34,83545.56,0.38,-0.18,0.13,0.05,2025-09-12 15:07:30+00:00,0.30,41813.28,0.11,18282.58,0.22,0.15,0.21,0.06


In [622]:
selected_wallets = wallet_cohorts["multi_filter"]
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)

In [623]:
WORKSPACE_DIR/"selected_wallets_v2.parquet"

PosixPath('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [624]:
# from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
# from polymarket_analysis.strategy.registry import save_strategy

# all_strategies = build_strategies_from_sweep(
#     wallet_cohorts=wallet_cohorts,
#     signal_threshold=SIGNAL_THRESHOLD,
#     selection_metric=BEST_SELECTION_METRIC,
#     top_n=BEST_TOP_N,
#     sweep_df=cohort_sweep,
#     extra_metadata={
#         "end_date_train": str(END_DATE_TRAIN),
#         "train_a_end_date": str(TRAIN_A_END_DATE),
#     },
# )

# for strategy in all_strategies:
#     parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
#     print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

# print(f"\nTotal strategies saved: {len(all_strategies)}")


## Strategy registry summary

In [625]:
# from polymarket_analysis.strategy.registry import load_all_strategies

# registry = load_all_strategies(WORKSPACE_DIR)
# summary_rows = []
# for sid, s in registry.items():
#     summary_rows.append({
#         "strategy_id": s.strategy_id,
#         "name": s.name,
#         "selection_method": s.selection_method,
#         "num_wallets": len(s.wallets),
#         "trigger_fn": s.trigger.fn_ref.split(".")[-1],
#         "threshold": s.trigger.params.get("threshold"),
#         "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
#     })

# pd.DataFrame(summary_rows)


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [626]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [627]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(3755374, 3135144, 620230)

In [628]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

7184

In [629]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

11998
6150


,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,...,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x134a63b764ac7b008356e8db1857db94e6b09e42,0x00f5b3584cb0cbf4441c697a179f5bbb72a4e3e30c3ea3d9ea6c026eb47b1895,2026-05-19 01:12:14+00:00,BUY,19.34,19.34,0.88,17.08,19.34,2.26,...,3.00,2026-05-31T00:00:00Z,"Will the Long Island Rail Road strike end by May 19, 2026?","[Politics, strike, New York City, Lirr]",Politics,14068254258217661656051444333727231232167965861089298160769372981036435245690,Yes,2.26,17.08,8.83
1,0x134a63b764ac7b008356e8db1857db94e6b09e42,0x00f5b3584cb0cbf4441c697a179f5bbb72a4e3e30c3ea3d9ea6c026eb47b1895,2026-05-19 01:12:17+00:00,BUY,48.49,19.15,0.92,17.68,19.15,1.47,...,2.00,2026-05-31T00:00:00Z,"Will the Long Island Rail Road strike end by May 19, 2026?","[Politics, strike, New York City, Lirr]",Politics,14068254258217661656051444333727231232167965861089298160769372981036435245690,Yes,1.47,17.68,12.44


In [630]:
pd.set_option('display.max_colwidth', 100)

In [631]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(10)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-04-16 00:15:00+00:00,0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5,0x1521b47bf0c41f6b7fd3ad41cdec566812c8f23e,BUY,474.68,778.17,1476.60,303.48,28,778.17,1,100.00,63.93,163.93
1,2026-04-16 00:20:00+00:00,0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5,0x1521b47bf0c41f6b7fd3ad41cdec566812c8f23e,BUY,222.46,364.69,1040.94,142.23,4,364.69,1,100.00,63.93,163.93
2,2026-04-16 00:50:00+00:00,0xbbc6689d0f6d57ea42168836712237c7308b3e0118c8914d31b6126d0f3254c5,0x481d26886bb2c3f106ca59def5d063a17fbe48cf,BUY,83.43,347.63,-192.00,-83.43,1,347.63,1,83.43,-83.43,347.63
3,2026-04-16 01:45:00+00:00,0x3c41f17860ead2faeb7b055dd768ebd7f11c0996756828ddb09acebf8b5fef22,0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,BUY,12.98,16.23,40.00,3.25,1,16.23,1,12.98,3.25,16.23
4,2026-04-16 01:50:00+00:00,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,35.97,42.32,30.00,6.35,1,42.32,1,35.97,6.35,42.32
5,2026-04-16 02:00:00+00:00,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,17.00,17.05,0.05,0.05,1,17.05,1,17.00,0.05,17.05
6,2026-04-16 02:05:00+00:00,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,166.18,166.68,0.50,0.50,1,166.68,1,100.00,0.30,100.30
7,2026-04-16 02:20:00+00:00,0x136f5a0c27a62cf9a2e40a4f48425e43d61b9571a53a2529372c0065f3218a73,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,54.56,56.60,50.23,2.05,3,56.60,1,54.56,2.05,56.60
8,2026-04-16 03:25:00+00:00,0xf616ba46f3bc1412a80824309a02b73ddbcd1a43a1f7a37aa161e54b812972d5,0x60a92c8620846d81f5ea17b0564e0d4b7c545a71,BUY,1180.76,1717.59,958.17,536.83,2,1717.59,1,100.00,45.46,145.46
9,2026-04-16 04:15:00+00:00,0xf3c44205f934fcb30f4fdd8af9e8ee5cd6ade1e587eb7aff5e8de2bdeeedacd3,0x60a92c8620846d81f5ea17b0564e0d4b7c545a71,BUY,8.10,62.31,-122.98,-8.10,1,62.31,1,8.10,-8.10,62.31


In [632]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [633]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [634]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [635]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).head(10)
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
condition_id,,,,,,
0xe546672750517f62c45a5a00067481981e62b9c20fa8220203232c9dc8fd2093,113886.59,208348.92,104031.62,61569.33,29,8
0x9967297556e4fd7f7e1e688a2e44d6fdbc42b0700576a1e4e67aa37764ac640a,17604.21,32423.71,10469.73,11364.40,19,9
0x1bb8bd43ada27e828ce693a5dc2896cd918b64869570761a61048ec78471ce9a,5840.20,16014.26,4800.72,5568.91,26,10
0x34f162f61a90524c7970782b6b64e04969ab1d68627fcda93d065aa49ad2abfb,11072.75,15433.38,8311.38,4310.63,14,2
0x57c1e8de9d359a76055fe1be95e46a1e72d0537811dcc2ccf070cdfa73d8ba33,19402.12,22893.92,29115.92,3491.80,21,10
0x3790655cbef2a9bb342b4b5186c5d2575719cdbcf171265479ec907f0794a654,16246.33,18782.13,3023.74,2319.48,9,1
0xa390a85e67d73fab31acf4af1ec3fea1102b3e745b1e9fceaa9d4b6e875aadaa,7373.76,9494.59,2738.91,2067.70,6,3
0x5c3e77051639ff3cc2cd1735d6667d15514da142558bc198bcd9e73d1feb2a7f,2870.00,4844.78,417.56,1826.26,7,2
0xb24aa72235c46c88613dca167f2e324aeb03e73f556fda680bbbdb815a5c40d0,12859.59,15555.39,4403.43,1767.81,36,11


In [636]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")